# Week 4 Guided Exercise - Solution

This notebook matches the student exercise. Every executable line is explained in plain English.

## Part A - Load and verify the tables

In [ ]:
# Import Pandas and use its standard short name.
import pandas as pd
# Read the long-format score records from the CSV file.
scores = pd.read_csv("student_scores.csv")
# Read the attendance records from the CSV file.
attendance = pd.read_csv("student_attendance.csv")
# Read the study-time records from the CSV file.
study_time = pd.read_csv("student_study_time.csv")
# Display the first five score records.
display(scores.head())
# Display the first five attendance records.
display(attendance.head())
# Display the first five study-time records.
display(study_time.head())

In [ ]:
# Display the shape of the score table.
print("Scores:", scores.shape)
# Display the shape of the attendance table.
print("Attendance:", attendance.shape)
# Display the shape of the study-time table.
print("Study time:", study_time.shape)
# Check whether attendance IDs are unique.
print("Attendance IDs unique:", attendance["student_id"].is_unique)
# Check whether study-time IDs are unique.
print("Study-time IDs unique:", study_time["student_id"].is_unique)

`student_id` repeats in the score table because each student has four measurements: Pre and Post scores for Math and Science.

## Part B - Group and aggregate

In [ ]:
# Calculate the mean score for each subject.
subject_means = scores.groupby("subject", as_index=False)["score"].mean()
# Group by subject and assessment and calculate named statistics.
score_summary = scores.groupby(["subject", "assessment"], as_index=False).agg(
    mean_score=("score", "mean"),
    median_score=("score", "median"),
    minimum_score=("score", "min"),
    maximum_score=("score", "max"),
    student_count=("student_id", "nunique"),
)
# Calculate the mean score for every grade and subject combination.
grade_subject_summary = scores.groupby(["grade", "subject"], as_index=False)["score"].mean()
# Display subject means.
display(subject_means)
# Display the multi-statistic summary.
display(score_summary)
# Display the grade and subject summary.
display(grade_subject_summary)

## Part C - Transform and pivot

In [ ]:
# Return each subject-assessment group mean to every original row.
scores["group_mean_score"] = scores.groupby(["subject", "assessment"])["score"].transform("mean")
# Calculate each score's distance from its group mean.
scores["difference_from_group_mean"] = scores["score"] - scores["group_mean_score"]
# Reshape the long scores into one row per student.
score_pivot = scores.pivot_table(index=["student_id", "name", "grade", "club"], columns=["subject", "assessment"], values="score", aggfunc="mean")
# Flatten the two-level column labels.
score_pivot.columns = [f"{subject.lower()}_{assessment.lower()}" for subject, assessment in score_pivot.columns]
# Return index labels to regular columns.
score_pivot = score_pivot.reset_index()
# Display the pivoted result.
display(score_pivot)

## Part D - Concatenate and investigate joins

In [ ]:
# Select Pre rows with the original score columns.
pre_scores = scores.loc[scores["assessment"] == "Pre", ["student_id", "name", "grade", "club", "subject", "assessment", "score"]]
# Select Post rows with the same columns.
post_scores = scores.loc[scores["assessment"] == "Post", ["student_id", "name", "grade", "club", "subject", "assessment", "score"]]
# Stack the compatible tables and create fresh row labels.
recombined_scores = pd.concat([pre_scores, post_scores], ignore_index=True)
# Select one row per score student.
score_students = score_pivot[["student_id", "name"]]
# Keep only IDs present in both tables.
inner_demo = score_students.merge(attendance, on="student_id", how="inner")
# Keep every ID from the score-student table.
left_demo = score_students.merge(attendance, on="student_id", how="left")
# Keep every ID from either table and label its source.
outer_demo = score_students.merge(attendance, on="student_id", how="outer", indicator=True)
# Print the concat and merge row counts.
print("Recombined:", len(recombined_scores), "Inner:", len(inner_demo), "Left:", len(left_demo), "Outer:", len(outer_demo))
# Display records that did not match in both tables.
display(outer_demo.loc[outer_demo["_merge"] != "both"])

`S13` appears only in attendance, so an inner merge removes it because no matching score-student key exists.

## Part E - Build a safe combined table

In [ ]:
# Match attendance to each score student and verify one-to-one keys.
student_analysis = score_pivot.merge(attendance, on="student_id", how="left", validate="one_to_one")
# Match study time to each score student and verify one-to-one keys.
student_analysis = student_analysis.merge(study_time, on="student_id", how="left", validate="one_to_one")
# Print the finished row count.
print("Final rows:", len(student_analysis))
# Display missing-value counts after merging.
display(student_analysis.isna().sum())
# Put student_id in the score table's index.
score_indexed = score_pivot.set_index("student_id")
# Put student_id in the study-time table's index.
study_indexed = study_time.set_index("student_id")
# Join study time to scores by matching index labels.
joined_demo = score_indexed.join(study_indexed, how="left", validate="one_to_one")
# Display the joined result.
display(joined_demo.head())

The table should retain 12 rows because the score table is the left table and has 12 unique students. The one-to-one validation prevents duplicate keys from multiplying rows.

## Part F - Engineer features

In [ ]:
# Calculate Math improvement.
student_analysis["math_improvement"] = student_analysis["math_post"] - student_analysis["math_pre"]
# Calculate Science improvement.
student_analysis["science_improvement"] = student_analysis["science_post"] - student_analysis["science_pre"]
# Calculate average improvement across both subjects.
student_analysis["average_improvement"] = student_analysis[["math_improvement", "science_improvement"]].mean(axis=1)
# Calculate the mean of both Post scores.
student_analysis["overall_post_score"] = student_analysis[["math_post", "science_post"]].mean(axis=1)
# Convert attendance to a percentage.
student_analysis["attendance_rate"] = student_analysis["days_present"] / student_analysis["days_possible"] * 100
# Calculate average hours in each study session.
student_analysis["hours_per_session"] = student_analysis["weekly_study_hours"] / student_analysis["weekly_sessions"]
# Round four summary features to one decimal place.
student_analysis[["average_improvement", "overall_post_score", "attendance_rate", "hours_per_session"]] = student_analysis[["average_improvement", "overall_post_score", "attendance_rate", "hours_per_session"]].round(1)
# Define attendance category boundaries.
attendance_bins = [0, 90, 95, 100]
# Define attendance category names.
attendance_labels = ["Needs Attention", "Good", "Excellent"]
# Convert attendance percentages to categories.
student_analysis["attendance_category"] = pd.cut(student_analysis["attendance_rate"], bins=attendance_bins, labels=attendance_labels, include_lowest=True)
# Flag rows meeting either review condition.
student_analysis["support_flag"] = (student_analysis["attendance_rate"] < 90) | (student_analysis["overall_post_score"] < 80)
# Convert Boolean flags into readable labels.
student_analysis["support_status"] = student_analysis["support_flag"].map({True: "Review", False: "On Track"})
# Display the engineered features.
display(student_analysis[["name", "average_improvement", "overall_post_score", "attendance_rate", "hours_per_session", "attendance_category", "support_status"]])

## Part G - Final report

In [ ]:
# Group the finished table by grade and calculate named summaries.
grade_report = student_analysis.groupby("grade", as_index=False).agg(
    students=("student_id", "nunique"),
    mean_post_score=("overall_post_score", "mean"),
    mean_improvement=("average_improvement", "mean"),
    mean_attendance=("attendance_rate", "mean"),
    mean_study_hours=("weekly_study_hours", "mean"),
)
# Round grade-level values to one decimal place.
grade_report = grade_report.round(1)
# Display the grade report.
display(grade_report)
# Select evidence columns for students marked Review.
review_students = student_analysis.loc[student_analysis["support_status"] == "Review", ["name", "attendance_rate", "overall_post_score", "support_status"]]
# Display students marked for review.
display(review_students)

## Sample conclusion

Grade-level summaries show differences in post scores, improvement, attendance, and study time, but each grade contains only four students. Hassan shows strong average improvement while also having an attendance rate below 90%, illustrating that individual patterns may differ from group averages. `attendance_rate` standardizes days present as a percentage, making students comparable. The final merge retained all 12 score students, and `validate="one_to_one"` confirmed that lookup-table keys did not multiply rows. The support flag is a transparent screening rule, not a diagnosis. Because the dataset is small and observational, it cannot show that attendance or study time caused score changes.